# Clean Raw Mercury

Reads from this notebook's `input/` folder and writes to its `output/` and `reports/` folders (all siblings of this notebook file). Each of the four sections below is self-contained (its own schema constants, `clean()` function, and `df_raw_*`/`df_cleaned_*` variables) and can be re-run independently without clobbering another section's results.

## Imports & shared helpers

`find_default_input`, `month_tag_from_filename`, `write_report`, `collapse_duplicate_rows`, `drop_blank_and_missing_key_rows`, `extract_duplicate_group_rows`, and `write_dropped_and_dupes` are identical in shape across all four source notebooks. Here they're defined once, parameterized by `directory`/`prefix`/`filename_re` (and `reports_dir`/dataframes for the report writer, `ls_cols`/`sum_cols`/`sort_cols`/`key_col` for the dedup/drop/audit helpers) instead of closing over notebook-global constants, and each of the four sections below calls these same functions with its own values.

Schema constants, `clean()` logic, and sort/rename rules genuinely differ per dataset and are kept written out separately in each section rather than hidden behind a generic abstraction.

In [1]:
import csv
import io
import re
from pathlib import Path

import pandas as pd

NOTEBOOK_DIR = Path.cwd()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
REPORTS_DIR = NOTEBOOK_DIR / "reports"
DROPPED_AND_DUPES_DIR = NOTEBOOK_DIR / "dropped_and_dupes"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
DROPPED_AND_DUPES_DIR.mkdir(parents=True, exist_ok=True)

### `find_default_input`

Looks for a single `{prefix}_yyyy-mm-dd.txt` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use.

In [2]:
def find_default_input(directory: Path, prefix: str, filename_re: re.Pattern) -> Path:
    matches = sorted(p for p in directory.glob(f"{prefix}_*.txt") if filename_re.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No {prefix}_yyyy-mm-dd.txt file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]

### `month_tag_from_filename`

The output name has the format `{prefix}_<month_tag>_cleaned.csv`, where `month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one). Independent of any raw `yearMonth` data column.

In [3]:
def month_tag_from_filename(path: Path, prefix: str, filename_re: re.Pattern) -> str:
    match = filename_re.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern {prefix}_yyyy-mm-dd.txt")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"

### `sum_numeric_cols`

Returns `{col: sum}` for a dataframe's numeric columns. Used to compute the BEFORE_SUM (right after blank/missing-key rows are dropped) and AFTER_SUM (after cleaning and deduping) values that `write_report` compares in the report.

In [4]:
def sum_numeric_cols(df: pd.DataFrame, sum_cols: list[str]) -> dict[str, int]:
    return {col: int(df[col].sum()) for col in sum_cols}

### `write_report`

Writes the plain-text summary report: input filename, raw row/duplicate counts, `month_tag`, blank/missing-key rows dropped, duplicate rows collapsed, cleaned row/duplicate counts, output filename, then (after three blank lines) `df_cleaned.describe()`. Duplicate counts use pandas' default `duplicated()` (`keep="first"`) â€” the number of rows that would go away if the dataframe were deduplicated. `rows_before_collapse` is the row count right after `clean_*()` but before `collapse_duplicate_rows()`, which is what lets the report split "blank/missing-key rows dropped" from "duplicate rows collapsed" instead of lumping both into one raw-vs-cleaned delta. Saved to `reports_dir` as `{prefix}_{month_tag}_report.txt`. Returns `(report_path, report_text)` so the calling cell can print/inspect it.

In [5]:
def write_report(
    reports_dir: Path,
    prefix: str,
    month_tag: str,
    input_path: Path,
    df_raw: pd.DataFrame,
    df_cleaned: pd.DataFrame,
    output_path: Path,
    rows_before_collapse: int,
    sums_before_collapse: dict[str, int],
    sums_after_collapse: dict[str, int],
) -> tuple[Path, str]:
    report_lines = [
        f"Input file: {input_path.name}",
        f"Raw row count: {len(df_raw)}",
        f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
        "=======================================================================",
        f"Month tag: {month_tag}",
        f"Blank/missing-key rows dropped: {len(df_raw) - rows_before_collapse}",
        f"Duplicate rows collapsed: {rows_before_collapse - len(df_cleaned)}",
        "=======================================================================",
        f"Cleaned row count: {len(df_cleaned)}",
        f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
        f"Output file: {output_path.name}",
        "=======================================================================",
        "Numeric field sums (BEFORE_SUM: after blank/missing-key rows dropped; "
        "AFTER_SUM: after cleaning and deduping):",
    ]
    for col in sums_before_collapse:
        before_sum = sums_before_collapse[col]
        after_sum = sums_after_collapse[col]
        match = "match" if before_sum == after_sum else "MISMATCH"
        report_lines.append(f"  {col}: BEFORE_SUM={before_sum}, AFTER_SUM={after_sum} ({match})")
    report_text = "\n".join(report_lines) + "\n"
    report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

    reports_dir.mkdir(parents=True, exist_ok=True)
    report_path = reports_dir / f"{prefix}_{month_tag}_report.txt"
    report_path.write_text(report_text, encoding="utf-8")

    return report_path, report_text

### `read_semicolon_csv_protecting_backslashes`

Used by the DailyEvents/MonthlyEvents sections, both of which need `engine="python", escapechar="\\"` to parse genuine `\"..\"` escapes around quoted phrases inside `SearchTerm`/`ContentTitle` (e.g. `\"daily wellness check-in\"`). Left unguarded, `escapechar` strips *every* backslash it precedes, not just ones before a quote â€” so a literal backslash in `CompanyCode`/`CompanyName` (e.g. a client named `TBWA\RAAD`) would be silently corrupted to `TBWARAAD`. This pre-processes the raw text to double any backslash *not* immediately followed by `"`, so `escapechar` only ever consumes genuine `\"` sequences and every other backslash survives intact.

In [6]:
def read_semicolon_csv_protecting_backslashes(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        raw_text = f.read()

    # Protect literal backslashes that aren't a genuine CSV \" escape by doubling them,
    # so escapechar only ever consumes actual \" sequences below.
    protected_text = re.sub(r'\\(?!")', r"\\\\", raw_text)

    return pd.read_csv(
        io.StringIO(protected_text), sep=";", engine="python", escapechar="\\",
        dtype=str, keep_default_na=False, encoding="utf-8",
    )

### `blank_out_dash_cells`

Replaces any cell whose value is exactly `"-"` (ignoring surrounding whitespace) with a blank string. Applied to every raw dataframe right after it's read, so a `"-"` placeholder for missing data is normalized to blank before any downstream cleaning runs — and the cleaned output CSVs end up with blank cells instead of `"-"` in the same spots.

In [7]:
def blank_out_dash_cells(df: pd.DataFrame) -> pd.DataFrame:
    return df.apply(lambda col: col.mask(col.str.strip() == "-", ""))

### `collapse_duplicate_rows`

Duplicates are not allowed in the final cleaned dataset. Each dataset's raw export can contain multiple rows that agree on every field except its numeric measure(s) (e.g. the same `Date`/`CompanyCode`/`EventType`/... combination reported twice with different `UniqueUsers` counts) â€” this groups by every `ls_cols` field *except* `sum_cols`, sums `sum_cols` within each group, and re-sorts by `sort_cols` afterward (grouping does not guarantee the output is already in `SORT_COLS_*` order). Called once per section, right after `clean_*()` and before the cleaned CSV/report are written, so both reflect the deduplicated data.

In [8]:
def collapse_duplicate_rows(
    df: pd.DataFrame, ls_cols: list[str], sum_cols: list[str], sort_cols: list[str]
) -> pd.DataFrame:
    group_cols = [c for c in ls_cols if c not in sum_cols]
    collapsed = df.groupby(group_cols, as_index=False, sort=False)[sum_cols].sum()[ls_cols]
    return collapsed.sort_values(by=sort_cols, ascending=True).reset_index(drop=True)

### `drop_blank_and_missing_key_rows`

Splits a renamed-but-not-yet-filtered dataframe into `(kept, dropped)`. A row is dropped if it's blank across every `ls_cols` field present in the raw data, or if it's missing the dataset's date key (`Date`/`MonthDate`) after that first blank check. The dropped rows are tagged `Status="dropped blank"` so they can be written to the `dropped_and_dupes/` audit CSV further down, alongside duplicate-group rows. Called once per section, right after the section's rename step and before any date parsing (which would fail on the blank date values these rows can carry).

In [9]:
def drop_blank_and_missing_key_rows(
    df: pd.DataFrame, ls_cols: list[str], key_col: str
) -> tuple[pd.DataFrame, pd.DataFrame]:
    present_ls_cols = [c for c in ls_cols if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    blank_rows = df.loc[is_blank]
    df = df.loc[~is_blank].copy()

    missing_key = df[key_col].str.strip() == ""
    missing_key_rows = df.loc[missing_key]
    df = df.loc[~missing_key].copy()

    dropped = pd.concat([blank_rows, missing_key_rows], ignore_index=True)[present_ls_cols].copy()
    dropped["Status"] = "dropped blank"
    return df, dropped

### `extract_duplicate_group_rows`

Identifies the rows `collapse_duplicate_rows` is about to merge, *before* merging them â€” grouping by the same `group_cols` (`ls_cols` minus `sum_cols`) that `collapse_duplicate_rows` uses, so the two stay in lockstep by construction. Uses `duplicated(subset=group_cols, keep=False)` to keep **every** row in a group of size > 1 (all N rows), not just the N-1 that would disappear on collapse. Tagged `Status="duplicate collapsed"`. Called on the fully-cleaned, pre-collapse dataframe â€” the same one `rows_before_collapse` is measured from.

In [10]:
def extract_duplicate_group_rows(df: pd.DataFrame, ls_cols: list[str], sum_cols: list[str]) -> pd.DataFrame:
    group_cols = [c for c in ls_cols if c not in sum_cols]
    dup_rows = df.loc[df.duplicated(subset=group_cols, keep=False), ls_cols].copy()
    dup_rows["Status"] = "duplicate collapsed"
    return dup_rows

### `write_dropped_and_dupes`

Combines a section's dropped-blank rows and duplicate-group rows into one `Status`-tagged audit dataframe (reindexed to `ls_cols + ["Status"]`) and writes it to `dropped_and_dupes/{prefix}_{month_tag}_dropped_and_dupes.csv`. Always writes the file, even when both inputs are empty, so every section produces a predictable audit CSV. Returns the output path.

In [11]:
def write_dropped_and_dupes(
    dropped_and_dupes_dir: Path, prefix: str, month_tag: str, ls_cols: list[str],
    dropped_blank: pd.DataFrame, duplicate_rows: pd.DataFrame,
) -> Path:
    combined = pd.concat([dropped_blank, duplicate_rows], ignore_index=True).reindex(columns=ls_cols + ["Status"])
    dropped_and_dupes_dir.mkdir(parents=True, exist_ok=True)
    path = dropped_and_dupes_dir / f"{prefix}_{month_tag}_dropped_and_dupes.csv"
    combined.to_csv(path, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)
    return path

## 1. MercuryDailyEvents

Cleans a raw `MercuryDailyEvents_yyyy-mm-dd.txt` export.

### Schema constants

In [12]:
LS_COLS_DE = [
    "Date", "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "DeviceCategory",
    "SearchTerm", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "UniqueUsers", "DateRange",
]
LS_STRING_COLS_DE = [
    "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "DeviceCategory",
    "SearchTerm", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "DateRange",
]
LS_INT_COLS_DE = ["UniqueUsers"]

RENAME_MAP_DE = {
    "ContentTitleEN": "ContentTitle",
    "downloadLanguage": "DownloadLanguage",
    "uniqueUsers": "UniqueUsers",
}

SORT_COLS_DE = [
    "Date", "CompanyCode", "CompanyName", "Country", "Operation", "EventType",
    "DeviceCategory", "SearchTerm", "ContentType", "ContentTitle",
]

PREFIX_DE = "MercuryDailyEvents"
FILENAME_RE_DE = re.compile(r"^MercuryDailyEvents_(\d{4})-(\d{2})-\d{2}\.txt$")

### Cleaning logic

1. Rename `ContentTitleEN`â†’`ContentTitle`, `downloadLanguage`â†’`DownloadLanguage`, `uniqueUsers`â†’`UniqueUsers`.
2. Drop rows that are blank across every `LS_COLS_DE` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS_DE`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DE` fields.
7. Cast `UniqueUsers` to integer type.
8. Sort ascending by `SORT_COLS_DE`.

In [13]:
def clean_de(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.rename(columns=RENAME_MAP_DE)

    df, dropped_de = drop_blank_and_missing_key_rows(df, LS_COLS_DE, "Date")

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS_DE]

    for col in LS_STRING_COLS_DE:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DE:
        df[col] = df[col].replace("", "0").astype(int)

    df = df.sort_values(by=SORT_COLS_DE, ascending=True).reset_index(drop=True)

    return df, dropped_de

### Configure the input file

Leave `INPUT_FILE_DE` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.

In [14]:
INPUT_FILE_DE = None  # e.g. "input/MercuryDailyEvents_2026-08-02.txt"

input_path_de = Path(INPUT_FILE_DE).resolve() if INPUT_FILE_DE else find_default_input(INPUT_DIR, PREFIX_DE, FILENAME_RE_DE)
month_tag_de = month_tag_from_filename(input_path_de, PREFIX_DE, FILENAME_RE_DE)
input_path_de, month_tag_de

(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/Mercury/input/MercuryDailyEvents_2026-09-02.txt'),
 '202608')

### Read the raw text file

Uses the shared `read_semicolon_csv_protecting_backslashes` helper (needs `escapechar` for the `\"..\"` escapes in `SearchTerm`/`ContentTitle`, protected against corrupting a literal backslash elsewhere in the row).

In [15]:
df_raw_de = read_semicolon_csv_protecting_backslashes(input_path_de)
df_raw_de = blank_out_dash_cells(df_raw_de)
df_raw_de.shape

(1750, 21)

### Apply the cleaning steps

In [16]:
df_cleaned_de, dropped_blank_de = clean_de(df_raw_de)
df_cleaned_de.head()

,Date,CompanyCode,CompanyName,Country,Operation,EventType,DeviceCategory,SearchTerm,ContentType,ContentTitle,DownloadLanguage,Theme,Route,UniqueUsers,DateRange
0,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,child,,,,,,1,2026-08-01 - 2026-08-31
1,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,money,,,,,,1,2026-08-01 - 2026-08-31
2,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,n/a,,,,,,1,2026-08-01 - 2026-08-31
3,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,self esteem,,,,,,1,2026-08-01 - 2026-08-31
4,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,sleep,,,,,,1,2026-08-01 - 2026-08-31


### Collapse duplicate rows

Groups by every `LS_COLS_DE` field except `UniqueUsers` and sums `UniqueUsers` within each group, collapsing duplicate rows into a single one (then re-sorts by `SORT_COLS_DE`, since grouping doesn't preserve the earlier sort order).

In [17]:
rows_before_collapse_de = len(df_cleaned_de)
sums_before_collapse_de = sum_numeric_cols(df_cleaned_de, LS_INT_COLS_DE)
duplicate_rows_de = extract_duplicate_group_rows(df_cleaned_de, LS_COLS_DE, LS_INT_COLS_DE)
df_cleaned_de = collapse_duplicate_rows(df_cleaned_de, LS_COLS_DE, LS_INT_COLS_DE, SORT_COLS_DE)
sums_after_collapse_de = sum_numeric_cols(df_cleaned_de, LS_INT_COLS_DE)

print(f"Duplicate rows collapsed: {rows_before_collapse_de - len(df_cleaned_de)}")
df_cleaned_de.head()

Duplicate rows collapsed: 0


,Date,CompanyCode,CompanyName,Country,Operation,EventType,DeviceCategory,SearchTerm,ContentType,ContentTitle,DownloadLanguage,Theme,Route,UniqueUsers,DateRange
0,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,child,,,,,,1,2026-08-01 - 2026-08-31
1,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,money,,,,,,1,2026-08-01 - 2026-08-31
2,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,n/a,,,,,,1,2026-08-01 - 2026-08-31
3,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,self esteem,,,,,,1,2026-08-01 - 2026-08-31
4,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,search,Desktop,sleep,,,,,,1,2026-08-01 - 2026-08-31


### Save the cleaned dataset

In [18]:
output_path_de = OUTPUT_DIR / f"{PREFIX_DE}_{month_tag_de}_cleaned.csv"
df_cleaned_de.to_csv(output_path_de, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_de)} rows -> {output_path_de}")

Cleaned 1750 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\output\MercuryDailyEvents_202608_cleaned.csv


### Save the dropped/duplicates audit

Writes every dropped-blank row and every row from a collapsed duplicate group (tagged by `Status`) to `dropped_and_dupes/`.

In [19]:
dropped_and_dupes_path_de = write_dropped_and_dupes(
    DROPPED_AND_DUPES_DIR, PREFIX_DE, month_tag_de, LS_COLS_DE, dropped_blank_de, duplicate_rows_de,
)

print(f"Dropped/dupes audit ({len(dropped_blank_de) + len(duplicate_rows_de)} rows) -> {dropped_and_dupes_path_de}")

Dropped/dupes audit (0 rows) -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\dropped_and_dupes\MercuryDailyEvents_202608_dropped_and_dupes.csv


### Write summary report

In [20]:
report_path_de, report_text_de = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DE,
    month_tag=month_tag_de,
    input_path=input_path_de,
    df_raw=df_raw_de,
    df_cleaned=df_cleaned_de,
    output_path=output_path_de,
    rows_before_collapse=rows_before_collapse_de,
    sums_before_collapse=sums_before_collapse_de,
    sums_after_collapse=sums_after_collapse_de,
)

print(report_text_de)
print(f"Report written -> {report_path_de}")

Input file: MercuryDailyEvents_2026-09-02.txt
Raw row count: 1750
Raw duplicate rows: 0
Month tag: 202608
Blank/missing-key rows dropped: 0
Duplicate rows collapsed: 0
Cleaned row count: 1750
Cleaned duplicate rows: 0
Output file: MercuryDailyEvents_202608_cleaned.csv
Numeric field sums (BEFORE_SUM: after blank/missing-key rows dropped; AFTER_SUM: after cleaning and deduping):
  UniqueUsers: BEFORE_SUM=1770, AFTER_SUM=1770 (match)



       UniqueUsers
count  1750.000000
mean      1.011429
std       0.130468
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       3.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\reports\MercuryDailyEvents_202608_report.txt


## 2. MercuryDailyUsers

Cleans a raw `MercuryDailyUsers_yyyy-mm-dd.txt` export.

### Schema constants

In [21]:
LS_COLS_DU = [
    "Date", "CompanyCode", "CompanyName", "Country", "Operation", "Sessions", "UniqueUsers", "DateRange",
]
LS_STRING_COLS_DU = [
    "CompanyCode", "CompanyName", "Country", "Operation", "DateRange",
]
LS_INT_COLS_DU = ["Sessions", "UniqueUsers"]

SORT_COLS_DU = ["Date", "CompanyCode", "CompanyName", "Country", "Operation"]

PREFIX_DU = "MercuryDailyUsers"
FILENAME_RE_DU = re.compile(r"^MercuryDailyUsers_(\d{4})-(\d{2})-\d{2}\.txt$")

### Cleaning logic

1. Rename every raw column so its first letter is capitalised (`sessions`â†’`Sessions`, `uniqueUsers`â†’`UniqueUsers`, ...) â€” via a lambda rather than an explicit rename map.
2. Drop rows that are blank across every `LS_COLS_DU` field present in the raw data.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds).
5. Reorder/drop columns to match `LS_COLS_DU`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_DU` fields.
7. Cast `Sessions`, `UniqueUsers` to integer type.
8. Sort ascending by `SORT_COLS_DU`.

In [22]:
def clean_du(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.rename(columns=lambda c: c[:1].upper() + c[1:] if c else c)

    df, dropped_du = drop_blank_and_missing_key_rows(df, LS_COLS_DU, "Date")

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS_DU]

    for col in LS_STRING_COLS_DU:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_DU:
        df[col] = df[col].replace("", "0").astype(int)

    df = df.sort_values(by=SORT_COLS_DU, ascending=True).reset_index(drop=True)

    return df, dropped_du

### Configure the input file

Leave `INPUT_FILE_DU` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.

In [23]:
INPUT_FILE_DU = None  # e.g. "input/MercuryDailyUsers_2026-08-02.txt"

input_path_du = Path(INPUT_FILE_DU).resolve() if INPUT_FILE_DU else find_default_input(INPUT_DIR, PREFIX_DU, FILENAME_RE_DU)
month_tag_du = month_tag_from_filename(input_path_du, PREFIX_DU, FILENAME_RE_DU)
input_path_du, month_tag_du

(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/Mercury/input/MercuryDailyUsers_2026-09-02.txt'),
 '202608')

### Read the raw text file

Plain read, no `engine`/`escapechar` â€” this file has no free-text content needing escaped quotes, and those options would only risk corrupting a literal backslash in `CompanyCode`/`CompanyName`.

In [24]:
df_raw_du = pd.read_csv(input_path_du, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw_du = blank_out_dash_cells(df_raw_du)
df_raw_du.shape

(194, 14)

### Apply the cleaning steps

In [25]:
df_cleaned_du, dropped_blank_du = clean_du(df_raw_du)
df_cleaned_du.head()

,Date,CompanyCode,CompanyName,Country,Operation,Sessions,UniqueUsers,DateRange
0,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,1,1,2026-08-01 - 2026-08-31
1,2026-08-03 00:00:00.000,BDMY,Becton Dickinson,Malta,Lyra Health Malaysia Sdn Bhd,2,1,2026-08-01 - 2026-08-31
2,2026-08-03 00:00:00.000,ICASSATEST,ICAS South Africa (Test Preview),United Kingdom,Lyra Southern Africa Pty Ltd,1,1,2026-08-01 - 2026-08-31
3,2026-08-03 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),South Africa,Lyra Health International Ltd,4,3,2026-08-01 - 2026-08-31
4,2026-08-03 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),United Kingdom,Lyra Health International Ltd,2,1,2026-08-01 - 2026-08-31


### Collapse duplicate rows

Groups by every `LS_COLS_DU` field except `Sessions`/`UniqueUsers` and sums `Sessions`, `UniqueUsers` within each group, collapsing duplicate rows into a single one (then re-sorts by `SORT_COLS_DU`, since grouping doesn't preserve the earlier sort order).

In [26]:
rows_before_collapse_du = len(df_cleaned_du)
sums_before_collapse_du = sum_numeric_cols(df_cleaned_du, LS_INT_COLS_DU)
duplicate_rows_du = extract_duplicate_group_rows(df_cleaned_du, LS_COLS_DU, LS_INT_COLS_DU)
df_cleaned_du = collapse_duplicate_rows(df_cleaned_du, LS_COLS_DU, LS_INT_COLS_DU, SORT_COLS_DU)
sums_after_collapse_du = sum_numeric_cols(df_cleaned_du, LS_INT_COLS_DU)

print(f"Duplicate rows collapsed: {rows_before_collapse_du - len(df_cleaned_du)}")
df_cleaned_du.head()

Duplicate rows collapsed: 0


,Date,CompanyCode,CompanyName,Country,Operation,Sessions,UniqueUsers,DateRange
0,2026-08-01 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),Malaysia,Lyra Health International Ltd,1,1,2026-08-01 - 2026-08-31
1,2026-08-03 00:00:00.000,BDMY,Becton Dickinson,Malta,Lyra Health Malaysia Sdn Bhd,2,1,2026-08-01 - 2026-08-31
2,2026-08-03 00:00:00.000,ICASSATEST,ICAS South Africa (Test Preview),United Kingdom,Lyra Southern Africa Pty Ltd,1,1,2026-08-01 - 2026-08-31
3,2026-08-03 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),South Africa,Lyra Health International Ltd,4,3,2026-08-01 - 2026-08-31
4,2026-08-03 00:00:00.000,ICASTESTCS,ICAS Client Services (Test Preview),United Kingdom,Lyra Health International Ltd,2,1,2026-08-01 - 2026-08-31


### Save the cleaned dataset

In [27]:
output_path_du = OUTPUT_DIR / f"{PREFIX_DU}_{month_tag_du}_cleaned.csv"
df_cleaned_du.to_csv(output_path_du, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_du)} rows -> {output_path_du}")

Cleaned 194 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\output\MercuryDailyUsers_202608_cleaned.csv


### Save the dropped/duplicates audit

Writes every dropped-blank row and every row from a collapsed duplicate group (tagged by `Status`) to `dropped_and_dupes/`.

In [28]:
dropped_and_dupes_path_du = write_dropped_and_dupes(
    DROPPED_AND_DUPES_DIR, PREFIX_DU, month_tag_du, LS_COLS_DU, dropped_blank_du, duplicate_rows_du,
)

print(f"Dropped/dupes audit ({len(dropped_blank_du) + len(duplicate_rows_du)} rows) -> {dropped_and_dupes_path_du}")

Dropped/dupes audit (0 rows) -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\dropped_and_dupes\MercuryDailyUsers_202608_dropped_and_dupes.csv


### Write summary report

In [29]:
report_path_du, report_text_du = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_DU,
    month_tag=month_tag_du,
    input_path=input_path_du,
    df_raw=df_raw_du,
    df_cleaned=df_cleaned_du,
    output_path=output_path_du,
    rows_before_collapse=rows_before_collapse_du,
    sums_before_collapse=sums_before_collapse_du,
    sums_after_collapse=sums_after_collapse_du,
)

print(report_text_du)
print(f"Report written -> {report_path_du}")

Input file: MercuryDailyUsers_2026-09-02.txt
Raw row count: 194
Raw duplicate rows: 0
Month tag: 202608
Blank/missing-key rows dropped: 0
Duplicate rows collapsed: 0
Cleaned row count: 194
Cleaned duplicate rows: 0
Output file: MercuryDailyUsers_202608_cleaned.csv
Numeric field sums (BEFORE_SUM: after blank/missing-key rows dropped; AFTER_SUM: after cleaning and deduping):
  Sessions: BEFORE_SUM=272, AFTER_SUM=272 (match)
  UniqueUsers: BEFORE_SUM=214, AFTER_SUM=214 (match)



         Sessions  UniqueUsers
count  194.000000   194.000000
mean     1.402062     1.103093
std      0.803616     0.380470
min      1.000000     1.000000
25%      1.000000     1.000000
50%      1.000000     1.000000
75%      2.000000     1.000000
max      5.000000     3.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\reports\MercuryDailyUsers_202608_report.txt


## 3. MercuryMonthlyEvents

Cleans a raw `MercuryMonthlyEvents_yyyy-mm-dd.txt` export.

### Schema constants

Note the column order here (`EventType, SearchTerm, DeviceCategory`) intentionally differs from DailyEvents' order (`EventType, DeviceCategory, SearchTerm`) â€” faithful to the original notebooks, not a typo.

In [30]:
LS_COLS_ME = [
    "MonthDate", "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "SearchTerm",
    "DeviceCategory", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "UniqueUsers", "DateRange",
]
LS_STRING_COLS_ME = [
    "CompanyCode", "CompanyName", "Country", "Operation", "EventType", "SearchTerm",
    "DeviceCategory", "ContentType", "ContentTitle", "DownloadLanguage", "Theme", "Route",
    "DateRange",
]
LS_INT_COLS_ME = ["UniqueUsers"]

RENAME_MAP_ME = {
    "yearMonth": "MonthDate",
    "ContentTitleEN": "ContentTitle",
    "downloadLanguage": "DownloadLanguage",
    "uniqueUsers": "UniqueUsers",
}

SORT_COLS_ME = ["MonthDate", "CompanyCode", "CompanyName", "Country", "Operation", "EventType"]

PREFIX_ME = "MercuryMonthlyEvents"
FILENAME_RE_ME = re.compile(r"^MercuryMonthlyEvents_(\d{4})-(\d{2})-\d{2}\.txt$")

### Cleaning logic

1. Rename `yearMonth`â†’`MonthDate`, `ContentTitleEN`â†’`ContentTitle`, `downloadLanguage`â†’`DownloadLanguage`, `uniqueUsers`â†’`UniqueUsers`.
2. Drop rows that are blank across every `LS_COLS_ME` field present in the raw data.
3. Drop rows where `MonthDate` is blank.
4. Reformat `MonthDate` (raw `"yyyy-mm"`, no day) to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds), defaulting to the 1st of the month.
5. Reorder/drop columns to match `LS_COLS_ME`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_ME` fields.
7. Cast `UniqueUsers` to integer type.
8. Sort ascending by `SORT_COLS_ME` (note: only 6 columns, unlike DailyEvents' 10 â€” preserved faithfully, not "fixed" to match).

In [31]:
def clean_me(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.rename(columns=RENAME_MAP_ME)

    df, dropped_me = drop_blank_and_missing_key_rows(df, LS_COLS_ME, "MonthDate")

    # Raw MonthDate values are "yyyy-mm" (no day); %m-only parsing defaults the day to 1.
    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["MonthDate"] = pd.to_datetime(df["MonthDate"], format="%Y-%m").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS_ME]

    for col in LS_STRING_COLS_ME:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_ME:
        df[col] = df[col].replace("", "0").astype(int)

    df = df.sort_values(by=SORT_COLS_ME, ascending=True).reset_index(drop=True)

    return df, dropped_me

### Configure the input file

Leave `INPUT_FILE_ME` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.

In [32]:
INPUT_FILE_ME = None  # e.g. "input/MercuryMonthlyEvents_2026-08-02.txt"

input_path_me = Path(INPUT_FILE_ME).resolve() if INPUT_FILE_ME else find_default_input(INPUT_DIR, PREFIX_ME, FILENAME_RE_ME)
month_tag_me = month_tag_from_filename(input_path_me, PREFIX_ME, FILENAME_RE_ME)
input_path_me, month_tag_me

(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/Mercury/input/MercuryMonthlyEvents_2026-09-02.txt'),
 '202608')

### Read the raw text file

Uses the shared `read_semicolon_csv_protecting_backslashes` helper (needs `escapechar` for the `\"..\"` escapes in `SearchTerm`/`ContentTitle`, protected against corrupting a literal backslash elsewhere in the row).

In [33]:
df_raw_me = read_semicolon_csv_protecting_backslashes(input_path_me)
df_raw_me = blank_out_dash_cells(df_raw_me)
df_raw_me.shape

(1594, 21)

### Apply the cleaning steps

In [34]:
df_cleaned_me, dropped_blank_me = clean_me(df_raw_me)
df_cleaned_me.head()

,MonthDate,CompanyCode,CompanyName,Country,Operation,EventType,SearchTerm,DeviceCategory,ContentType,ContentTitle,DownloadLanguage,Theme,Route,UniqueUsers,DateRange
0,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,Webinar Clip: Nature and Our Mental Health,en,,Search,1,2026-08-01 - 2026-08-31
1,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,How to Request a Consultation,en,,Search,1,2026-08-01 - 2026-08-31
2,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,Webinar Clip: Emotional Regulation,en,,Search,1,2026-08-01 - 2026-08-31
3,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,Exercises to Help Build Money Mindfulness,en,,Search,1,2026-08-01 - 2026-08-31
4,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,Mental health support: What happens when you c...,en,,Search,1,2026-08-01 - 2026-08-31


### Collapse duplicate rows

Groups by every `LS_COLS_ME` field except `UniqueUsers` and sums `UniqueUsers` within each group, collapsing duplicate rows into a single one (then re-sorts by `SORT_COLS_ME`, since grouping doesn't preserve the earlier sort order).

In [35]:
rows_before_collapse_me = len(df_cleaned_me)
sums_before_collapse_me = sum_numeric_cols(df_cleaned_me, LS_INT_COLS_ME)
duplicate_rows_me = extract_duplicate_group_rows(df_cleaned_me, LS_COLS_ME, LS_INT_COLS_ME)
df_cleaned_me = collapse_duplicate_rows(df_cleaned_me, LS_COLS_ME, LS_INT_COLS_ME, SORT_COLS_ME)
sums_after_collapse_me = sum_numeric_cols(df_cleaned_me, LS_INT_COLS_ME)

print(f"Duplicate rows collapsed: {rows_before_collapse_me - len(df_cleaned_me)}")
df_cleaned_me.head()

Duplicate rows collapsed: 0


,MonthDate,CompanyCode,CompanyName,Country,Operation,EventType,SearchTerm,DeviceCategory,ContentType,ContentTitle,DownloadLanguage,Theme,Route,UniqueUsers,DateRange
0,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,Webinar Clip: Nature and Our Mental Health,en,,Search,1,2026-08-01 - 2026-08-31
1,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,How to Request a Consultation,en,,Search,1,2026-08-01 - 2026-08-31
2,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,Webinar Clip: Emotional Regulation,en,,Search,1,2026-08-01 - 2026-08-31
3,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,Exercises to Help Build Money Mindfulness,en,,Search,1,2026-08-01 - 2026-08-31
4,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,download,,Desktop,Video,Mental health support: What happens when you c...,en,,Search,1,2026-08-01 - 2026-08-31


### Save the cleaned dataset

In [36]:
output_path_me = OUTPUT_DIR / f"{PREFIX_ME}_{month_tag_me}_cleaned.csv"
df_cleaned_me.to_csv(output_path_me, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_me)} rows -> {output_path_me}")

Cleaned 1594 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\output\MercuryMonthlyEvents_202608_cleaned.csv


### Save the dropped/duplicates audit

Writes every dropped-blank row and every row from a collapsed duplicate group (tagged by `Status`) to `dropped_and_dupes/`.

In [37]:
dropped_and_dupes_path_me = write_dropped_and_dupes(
    DROPPED_AND_DUPES_DIR, PREFIX_ME, month_tag_me, LS_COLS_ME, dropped_blank_me, duplicate_rows_me,
)

print(f"Dropped/dupes audit ({len(dropped_blank_me) + len(duplicate_rows_me)} rows) -> {dropped_and_dupes_path_me}")

Dropped/dupes audit (0 rows) -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\dropped_and_dupes\MercuryMonthlyEvents_202608_dropped_and_dupes.csv


### Write summary report

In [38]:
report_path_me, report_text_me = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_ME,
    month_tag=month_tag_me,
    input_path=input_path_me,
    df_raw=df_raw_me,
    df_cleaned=df_cleaned_me,
    output_path=output_path_me,
    rows_before_collapse=rows_before_collapse_me,
    sums_before_collapse=sums_before_collapse_me,
    sums_after_collapse=sums_after_collapse_me,
)

print(report_text_me)
print(f"Report written -> {report_path_me}")

Input file: MercuryMonthlyEvents_2026-09-02.txt
Raw row count: 1594
Raw duplicate rows: 0
Month tag: 202608
Blank/missing-key rows dropped: 0
Duplicate rows collapsed: 0
Cleaned row count: 1594
Cleaned duplicate rows: 0
Output file: MercuryMonthlyEvents_202608_cleaned.csv
Numeric field sums (BEFORE_SUM: after blank/missing-key rows dropped; AFTER_SUM: after cleaning and deduping):
  UniqueUsers: BEFORE_SUM=1630, AFTER_SUM=1630 (match)



       UniqueUsers
count  1594.000000
mean      1.022585
std       0.239254
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       8.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\reports\MercuryMonthlyEvents_202608_report.txt


## 4. MercuryMonthlyUsers

Cleans a raw `MercuryMonthlyUsers_yyyy-mm-dd.txt` export.

### Schema constants

In [39]:
LS_COLS_MU = ["MonthDate", "CompanyCode", "CompanyName", "Country", "Operation", "Sessions", "UniqueUsers", "DateRange"]
LS_STRING_COLS_MU = ["CompanyCode", "CompanyName", "Country", "Operation", "DateRange"]
LS_INT_COLS_MU = ["Sessions", "UniqueUsers"]

RENAME_MAP_MU = {
    "yearMonth": "MonthDate",
    "sessions": "Sessions",
    "uniqueUsers": "UniqueUsers",
}

SORT_COLS_MU = ["MonthDate", "CompanyCode", "CompanyName", "Country", "Operation"]

PREFIX_MU = "MercuryMonthlyUsers"
FILENAME_RE_MU = re.compile(r"^MercuryMonthlyUsers_(\d{4})-(\d{2})-\d{2}\.txt$")

### Cleaning logic

1. Rename `yearMonth`â†’`MonthDate`, `sessions`â†’`Sessions`, `uniqueUsers`â†’`UniqueUsers`.
2. Drop rows that are blank across every `LS_COLS_MU` field present in the raw data.
3. Drop rows where `MonthDate` is blank.
4. Reformat `MonthDate` (raw `"yyyy-mm"`, no day) to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds), defaulting to the 1st of the month.
5. Reorder/drop columns to match `LS_COLS_MU`.
6. Trim surrounding whitespace, then collapse any internal run of whitespace to a single space, on the `LS_STRING_COLS_MU` fields.
7. Cast `Sessions`, `UniqueUsers` to integer type.
8. Sort ascending by `SORT_COLS_MU`.

In [40]:
def clean_mu(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.rename(columns=RENAME_MAP_MU)

    df, dropped_mu = drop_blank_and_missing_key_rows(df, LS_COLS_MU, "MonthDate")

    # Raw MonthDate values are "yyyy-mm" (no day); %m-only parsing defaults the day to 1.
    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["MonthDate"] = pd.to_datetime(df["MonthDate"], format="%Y-%m").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[LS_COLS_MU]

    for col in LS_STRING_COLS_MU:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS_MU:
        df[col] = df[col].replace("", "0").astype(int)

    df = df.sort_values(by=SORT_COLS_MU, ascending=True).reset_index(drop=True)

    return df, dropped_mu

### Configure the input file

Leave `INPUT_FILE_MU` as `None` to auto-detect the single raw file in `input/`, or set it to an explicit path to override.

In [41]:
INPUT_FILE_MU = None  # e.g. "input/MercuryMonthlyUsers_2026-08-02.txt"

input_path_mu = Path(INPUT_FILE_MU).resolve() if INPUT_FILE_MU else find_default_input(INPUT_DIR, PREFIX_MU, FILENAME_RE_MU)
month_tag_mu = month_tag_from_filename(input_path_mu, PREFIX_MU, FILENAME_RE_MU)
input_path_mu, month_tag_mu

(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/Mercury/input/MercuryMonthlyUsers_2026-09-02.txt'),
 '202608')

### Read the raw text file

Plain read, no `engine`/`escapechar` â€” this file has no free-text content needing escaped quotes, and those options would only risk corrupting a literal backslash in `CompanyCode`/`CompanyName`.

In [42]:
df_raw_mu = pd.read_csv(input_path_mu, sep=";", dtype=str, keep_default_na=False, encoding="utf-8")
df_raw_mu = blank_out_dash_cells(df_raw_mu)
df_raw_mu.shape

(88, 14)

### Apply the cleaning steps

In [43]:
df_cleaned_mu, dropped_blank_mu = clean_mu(df_raw_mu)
df_cleaned_mu.head()

,MonthDate,CompanyCode,CompanyName,Country,Operation,Sessions,UniqueUsers,DateRange
0,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,6,1,2026-08-01 - 2026-08-31
1,2026-08-01 00:00:00.000,ABPIE,ABP - Ireland,Ireland,Lyra UK & Ireland Ltd,6,2,2026-08-01 - 2026-08-31
2,2026-08-01 00:00:00.000,ASAHI,ASAHI,Austria,Lyra Health International Ltd,2,1,2026-08-01 - 2026-08-31
3,2026-08-01 00:00:00.000,ATLAS,Atlas,Malta,Lyra Health International Ltd,2,2,2026-08-01 - 2026-08-31
4,2026-08-01 00:00:00.000,AVANTOR,Avantor,United States,Lyra Health International Ltd,1,1,2026-08-01 - 2026-08-31


### Collapse duplicate rows

Groups by every `LS_COLS_MU` field except `Sessions`/`UniqueUsers` and sums `Sessions`, `UniqueUsers` within each group, collapsing duplicate rows into a single one (then re-sorts by `SORT_COLS_MU`, since grouping doesn't preserve the earlier sort order).

In [44]:
rows_before_collapse_mu = len(df_cleaned_mu)
sums_before_collapse_mu = sum_numeric_cols(df_cleaned_mu, LS_INT_COLS_MU)
duplicate_rows_mu = extract_duplicate_group_rows(df_cleaned_mu, LS_COLS_MU, LS_INT_COLS_MU)
df_cleaned_mu = collapse_duplicate_rows(df_cleaned_mu, LS_COLS_MU, LS_INT_COLS_MU, SORT_COLS_MU)
sums_after_collapse_mu = sum_numeric_cols(df_cleaned_mu, LS_INT_COLS_MU)

print(f"Duplicate rows collapsed: {rows_before_collapse_mu - len(df_cleaned_mu)}")
df_cleaned_mu.head()

Duplicate rows collapsed: 0


,MonthDate,CompanyCode,CompanyName,Country,Operation,Sessions,UniqueUsers,DateRange
0,2026-08-01 00:00:00.000,ABL001,African Bank Limited,South Africa,Lyra Southern Africa Pty Ltd,6,1,2026-08-01 - 2026-08-31
1,2026-08-01 00:00:00.000,ABPIE,ABP - Ireland,Ireland,Lyra UK & Ireland Ltd,6,2,2026-08-01 - 2026-08-31
2,2026-08-01 00:00:00.000,ASAHI,ASAHI,Austria,Lyra Health International Ltd,2,1,2026-08-01 - 2026-08-31
3,2026-08-01 00:00:00.000,ATLAS,Atlas,Malta,Lyra Health International Ltd,2,2,2026-08-01 - 2026-08-31
4,2026-08-01 00:00:00.000,AVANTOR,Avantor,United States,Lyra Health International Ltd,1,1,2026-08-01 - 2026-08-31


### Save the cleaned dataset

In [45]:
output_path_mu = OUTPUT_DIR / f"{PREFIX_MU}_{month_tag_mu}_cleaned.csv"
df_cleaned_mu.to_csv(output_path_mu, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned_mu)} rows -> {output_path_mu}")

Cleaned 88 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\output\MercuryMonthlyUsers_202608_cleaned.csv


### Save the dropped/duplicates audit

Writes every dropped-blank row and every row from a collapsed duplicate group (tagged by `Status`) to `dropped_and_dupes/`.

In [46]:
dropped_and_dupes_path_mu = write_dropped_and_dupes(
    DROPPED_AND_DUPES_DIR, PREFIX_MU, month_tag_mu, LS_COLS_MU, dropped_blank_mu, duplicate_rows_mu,
)

print(f"Dropped/dupes audit ({len(dropped_blank_mu) + len(duplicate_rows_mu)} rows) -> {dropped_and_dupes_path_mu}")

Dropped/dupes audit (0 rows) -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\dropped_and_dupes\MercuryMonthlyUsers_202608_dropped_and_dupes.csv


### Write summary report

In [47]:
report_path_mu, report_text_mu = write_report(
    reports_dir=REPORTS_DIR,
    prefix=PREFIX_MU,
    month_tag=month_tag_mu,
    input_path=input_path_mu,
    df_raw=df_raw_mu,
    df_cleaned=df_cleaned_mu,
    output_path=output_path_mu,
    rows_before_collapse=rows_before_collapse_mu,
    sums_before_collapse=sums_before_collapse_mu,
    sums_after_collapse=sums_after_collapse_mu,
)

print(report_text_mu)
print(f"Report written -> {report_path_mu}")

Input file: MercuryMonthlyUsers_2026-09-02.txt
Raw row count: 88
Raw duplicate rows: 0
Month tag: 202608
Blank/missing-key rows dropped: 0
Duplicate rows collapsed: 0
Cleaned row count: 88
Cleaned duplicate rows: 0
Output file: MercuryMonthlyUsers_202608_cleaned.csv
Numeric field sums (BEFORE_SUM: after blank/missing-key rows dropped; AFTER_SUM: after cleaning and deduping):
  Sessions: BEFORE_SUM=272, AFTER_SUM=272 (match)
  UniqueUsers: BEFORE_SUM=113, AFTER_SUM=113 (match)



        Sessions  UniqueUsers
count  88.000000    88.000000
mean    3.090909     1.284091
std     5.391855     0.934017
min     1.000000     1.000000
25%     1.000000     1.000000
50%     1.000000     1.000000
75%     3.000000     1.000000
max    40.000000     8.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\Mercury\reports\MercuryMonthlyUsers_202608_report.txt


## Summary

Convenience recap of everything produced by this run.

In [48]:
print("Cleaned outputs:")
for label, out_path, rpt_path, dd_path in [
    ("DailyEvents",   output_path_de, report_path_de, dropped_and_dupes_path_de),
    ("DailyUsers",    output_path_du, report_path_du, dropped_and_dupes_path_du),
    ("MonthlyEvents", output_path_me, report_path_me, dropped_and_dupes_path_me),
    ("MonthlyUsers",  output_path_mu, report_path_mu, dropped_and_dupes_path_mu),
]:
    print(f"  {label:14s} -> {out_path.name}  (report: {rpt_path.name})  (dropped/dupes: {dd_path.name})")

Cleaned outputs:
  DailyEvents    -> MercuryDailyEvents_202608_cleaned.csv  (report: MercuryDailyEvents_202608_report.txt)  (dropped/dupes: MercuryDailyEvents_202608_dropped_and_dupes.csv)
  DailyUsers     -> MercuryDailyUsers_202608_cleaned.csv  (report: MercuryDailyUsers_202608_report.txt)  (dropped/dupes: MercuryDailyUsers_202608_dropped_and_dupes.csv)
  MonthlyEvents  -> MercuryMonthlyEvents_202608_cleaned.csv  (report: MercuryMonthlyEvents_202608_report.txt)  (dropped/dupes: MercuryMonthlyEvents_202608_dropped_and_dupes.csv)
  MonthlyUsers   -> MercuryMonthlyUsers_202608_cleaned.csv  (report: MercuryMonthlyUsers_202608_report.txt)  (dropped/dupes: MercuryMonthlyUsers_202608_dropped_and_dupes.csv)
